In [1]:
import pandas as pd

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from sklearn.pipeline import Pipeline
import joblib

print("="*80)
print("🏥 ULTIMATE CKD RISK MODEL")
print("   (Advanced Feature Engineering + Optimized Hyperparameters)")
print("="*80)

# Load dataset
df = pd.read_csv("Chronic_Kidney_Dsease_data.csv")
print(f"\n✓ Dataset loaded: {len(df)} samples")

# Create family history
df['FamilyHistory_Any'] = (
    df['FamilyHistoryKidneyDisease'] |
    df['FamilyHistoryHypertension'] |
    df['FamilyHistoryDiabetes']
).astype(int)


GFR_33 = df['GFR'].quantile(0.33)  # 48.7
GFR_67 = df['GFR'].quantile(0.67)  # 84.1
ACR_33 = df['ACR'].quantile(0.33)  # 100.7
ACR_67 = df['ACR'].quantile(0.67)  # 198.8

def percentile_based_classification(row):
    """Optimal labeling - 29% Low, 33% Med, 38% High"""
    score = 0
    
    # GFR (Primary)
    gfr = row['GFR']
    if gfr >= GFR_67:
        score += 0
    elif gfr >= GFR_33:
        score += 5
    else:
        score += 10
    
    # ACR (Secondary)
    acr = row['ACR']
    if acr < ACR_33:
        score += 0
    elif acr < ACR_67:
        score += 3
    else:
        score += 6
    
    # Age
    if row['Age'] >= 65:
        score += 2
    elif row['Age'] >= 50:
        score += 1
    
    # Comorbidities
    if row['AntidiabeticMedications'] == 1:
        score += 1
    if row['SystolicBP'] >= 150 or row['DiastolicBP'] >= 95:
        score += 1
    
    # Classification
    if score <= 6:
        return 0
    elif score <= 11:
        return 1
    else:
        return 2

df['RiskLevel'] = df.apply(percentile_based_classification, axis=1)

print(f"\n📊 Risk Distribution:")
class_counts = df['RiskLevel'].value_counts().sort_index()
for level, name in enumerate(['Low', 'Medium', 'High']):
    count = class_counts[level]
    pct = count / len(df) * 100
    print(f"  {name:6s}: {count:4d} ({pct:5.1f}%)")



print("\n🔧 Engineering Features...")

# 1. GFR Category (CKD Staging)
df['GFR_Category'] = pd.cut(df['GFR'], 
                             bins=[0, 30, 45, 60, 90, 150], 
                             labels=[4, 3, 2, 1, 0]).astype(int)

# 2. ACR Category (Albuminuria Staging)
df['ACR_Category'] = pd.cut(df['ACR'], 
                             bins=[0, 30, 300, 1000], 
                             labels=[0, 1, 2]).astype(int)

# 3. Age Groups
df['Age_Group'] = pd.cut(df['Age'], 
                          bins=[0, 40, 55, 70, 100], 
                          labels=[0, 1, 2, 3]).astype(int)

# 4. BMI Categories
df['BMI_Category'] = pd.cut(df['BMI'], 
                             bins=[0, 18.5, 25, 30, 50], 
                             labels=[0, 1, 2, 3]).astype(int)

# 5. Blood Pressure Categories
df['BP_Category'] = 0
df.loc[(df['SystolicBP'] >= 120) | (df['DiastolicBP'] >= 80), 'BP_Category'] = 1
df.loc[(df['SystolicBP'] >= 140) | (df['DiastolicBP'] >= 90), 'BP_Category'] = 2
df.loc[(df['SystolicBP'] >= 160) | (df['DiastolicBP'] >= 100), 'BP_Category'] = 3

# 6. Kidney Function Score (Composite)
df['Kidney_Function_Score'] = (df['GFR'] / 120) - (df['ACR'] / 300)

# 7. Risk Factor Count
df['Risk_Factor_Count'] = (
    (df['Smoking'] > 0).astype(int) +
    (df['AlcoholConsumption'] > 5).astype(int) +
    (df['BMI'] > 30).astype(int) +
    (df['PhysicalActivity'] < 3).astype(int) +
    df['FamilyHistory_Any'] +
    df['AntidiabeticMedications']
)

# 8. Symptom Score
df['Symptom_Score'] = df['Edema'] + df['UrinaryTractInfections'] + (df['NauseaVomiting'] > 0).astype(int)

# 9. Age × GFR Interaction (Older with poor GFR = very high risk)
df['Age_GFR_Interaction'] = df['Age'] * (120 - df['GFR']) / 100

# 10. GFR × ACR Product (Combined kidney damage)
df['GFR_ACR_Product'] = df['GFR'] * df['ACR'] / 1000

print("✅ 10 engineered features created")

# All features (19 original + 10 engineered = 29 total)
features = [
    # Original features (19)
    'Age', 'Gender', 'SystolicBP', 'DiastolicBP', 'BMI', 'Smoking',
    'AlcoholConsumption', 'PhysicalActivity', 'Edema', 'UrinaryTractInfections',
    'NauseaVomiting', 'FamilyHistory_Any', 'AntidiabeticMedications',
    'SerumCreatinine', 'BUNLevels', 'HemoglobinLevels', 'SerumElectrolytesSodium',
    'GFR', 'ACR',
    # Engineered features (10)
    'GFR_Category', 'ACR_Category', 'Age_Group', 'BMI_Category', 'BP_Category',
    'Kidney_Function_Score', 'Risk_Factor_Count', 'Symptom_Score',
    'Age_GFR_Interaction', 'GFR_ACR_Product'
]

print(f"📊 Total features: {len(features)} (19 original + 10 engineered)")

X = df[features]
y = df['RiskLevel']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

print(f"\n📋 Data Split: Train={len(X_train)}, Test={len(X_test)}")
print("="*80)



print("\n🔄 Training optimized Random Forest...")

pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler()),
    ('model', RandomForestClassifier(
        # Optimized hyperparameters from grid search
        n_estimators=130,              
        max_depth=7,                   
        min_samples_split=18,          
        min_samples_leaf=7,            
        max_samples=0.72,              
        min_impurity_decrease=0.0002,  
        max_features='sqrt',           
        class_weight='balanced',       
        bootstrap=True,
        oob_score=True,
        random_state=42,
        n_jobs=-1
    ))
])

# Train
pipeline.fit(X_train, y_train)
print("✅ Training complete!")

oob_score = pipeline.named_steps['model'].oob_score_
print(f"📊 OOB Score: {oob_score:.4f} ({oob_score*100:.1f}%)")
print("="*80)

# Evaluate
y_train_pred = pipeline.predict(X_train)
y_test_pred = pipeline.predict(X_test)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)
gap = train_acc - test_acc

print("\n📊 MODEL PERFORMANCE:")
print("="*80)
print(f"Training Accuracy:   {train_acc:.4f} ({train_acc*100:.1f}%)")
print(f"OOB Accuracy:        {oob_score:.4f} ({oob_score*100:.1f}%)")
print(f"Testing Accuracy:    {test_acc:.4f} ({test_acc*100:.1f}%)")
print(f"Train-Test Gap:      {gap:.4f} ({gap*100:.1f}%)")
print()

if gap <= 0.03:
    print("✅✅✅ PERFECT: Gap ≤ 3%!")
elif gap <= 0.05:
    print("✅ EXCELLENT: Gap ≤ 5%")
else:
    print("⚠️  WARNING: Gap > 5%")
print("="*80)

# Test results
print("\n🎯 TEST SET RESULTS:")
print("="*80)
cm = confusion_matrix(y_test, y_test_pred)
print("Confusion Matrix:")
print("         Predicted →")
print("Actual ↓   Low  Med  High")
for i, label in enumerate(['Low', 'Medium', 'High']):
    print(f"{label:6s}    {cm[i, 0]:3d}  {cm[i, 1]:3d}   {cm[i, 2]:3d}")

print("\n" + "-"*80)
print(classification_report(
    y_test, y_test_pred, 
    target_names=['Low', 'Medium', 'High'], 
    digits=3
))

# Per-class metrics
precision, recall, f1, support = precision_recall_fscore_support(
    y_test, y_test_pred, average=None
)

print("="*80)
print("📈 PER-CLASS PERFORMANCE:")
print("="*80)
for i, label in enumerate(['Low', 'Medium', 'High']):
    per_class_acc = cm[i, i] / support[i]
    print(f"{label:6s}: Precision={precision[i]:.1%}, Recall={recall[i]:.1%}, F1={f1[i]:.3f}, Accuracy={per_class_acc:.1%}")

min_precision = min(precision)
min_recall = min(recall)
min_f1 = min(f1)
min_per_class_acc = min((cm[i,i]/support[i]) for i in range(3))

print("\n" + "="*80)
print("✅ QUALITY GATES:")
print("="*80)
print(f"  Min Precision:       {min_precision:.1%} {'✅' if min_precision >= 0.85 else '⚠️' if min_precision >= 0.83 else '❌'} (Target: ≥85%)")
print(f"  Min Recall:          {min_recall:.1%} {'✅' if min_recall >= 0.85 else '⚠️' if min_recall >= 0.83 else '❌'} (Target: ≥85%)")
print(f"  Min F1-Score:        {min_f1:.3f} {'✅' if min_f1 >= 0.850 else '⚠️' if min_f1 >= 0.830 else '❌'} (Target: ≥0.850)")
print(f"  Min Per-Class Acc:   {min_per_class_acc:.1%} {'✅' if min_per_class_acc >= 0.85 else '⚠️' if min_per_class_acc >= 0.83 else '❌'} (Target: ≥85%)")
print(f"  Train-Test Gap:      {gap:.1%} {'✅' if gap <= 0.03 else '⚠️' if gap <= 0.05 else '❌'} (Target: ≤3%)")
print("="*80)

# Cross-validation
print("\n🔄 CROSS-VALIDATION (5-Fold Stratified):")
print("="*80)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
print(f"Fold Scores: {[f'{s:.3f}' for s in cv_scores]}")
print(f"Mean CV:  {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Stability: {'✅ Excellent' if cv_scores.std() < 0.02 else '⚠️ Check variance'}")
print("="*80)

# Feature importance
print("\n🔍 TOP 10 MOST IMPORTANT FEATURES:")
print("-"*80)
importances = pipeline.named_steps['model'].feature_importances_
feature_importance = sorted(zip(features, importances), key=lambda x: x[1], reverse=True)
for i, (feat, imp) in enumerate(feature_importance[:10], 1):
    print(f"{i:2d}. {feat:25s} {imp:.4f}")
print("="*80)

# Save model
joblib.dump(pipeline, 'ckd_risk_model.pkl')
print("\n✅ Model saved as 'ckd_risk_model.pkl'")


def predict_risk(user_input):
    """
    Predict CKD risk - UNCHANGED interface for FastAPI/Flutter compatibility
    Note: Engineered features are auto-created inside the prediction function
    """
    loaded_model = joblib.load('ckd_risk_model.pkl')
    input_df = pd.DataFrame([user_input])
    
    # Add missing original features with NaN for imputation
    for col in features[:19]: 
        if col not in input_df.columns:
            input_df[col] = np.nan
    
    # Create engineered features (same logic as training)
    input_df['GFR_Category'] = pd.cut(input_df['GFR'], bins=[0, 30, 45, 60, 90, 150], labels=[4, 3, 2, 1, 0]).astype(float)
    input_df['ACR_Category'] = pd.cut(input_df['ACR'], bins=[0, 30, 300, 1000], labels=[0, 1, 2]).astype(float)
    input_df['Age_Group'] = pd.cut(input_df['Age'], bins=[0, 40, 55, 70, 100], labels=[0, 1, 2, 3]).astype(float)
    input_df['BMI_Category'] = pd.cut(input_df['BMI'], bins=[0, 18.5, 25, 30, 50], labels=[0, 1, 2, 3]).astype(float)
    
    input_df['BP_Category'] = 0
    if ('SystolicBP' in input_df and input_df['SystolicBP'].notna().any()) or \
       ('DiastolicBP' in input_df and input_df['DiastolicBP'].notna().any()):
        if (input_df['SystolicBP'].fillna(0) >= 120).any() or (input_df['DiastolicBP'].fillna(0) >= 80).any():
            input_df['BP_Category'] = 1
        if (input_df['SystolicBP'].fillna(0) >= 140).any() or (input_df['DiastolicBP'].fillna(0) >= 90).any():
            input_df['BP_Category'] = 2
        if (input_df['SystolicBP'].fillna(0) >= 160).any() or (input_df['DiastolicBP'].fillna(0) >= 100).any():
            input_df['BP_Category'] = 3
    
    input_df['Kidney_Function_Score'] = (input_df['GFR'].fillna(60) / 120) - (input_df['ACR'].fillna(150) / 300)
    input_df['Risk_Factor_Count'] = (
        (input_df['Smoking'].fillna(0) > 0).astype(int) +
        (input_df['AlcoholConsumption'].fillna(0) > 5).astype(int) +
        (input_df['BMI'].fillna(25) > 30).astype(int) +
        (input_df['PhysicalActivity'].fillna(3) < 3).astype(int) +
        input_df['FamilyHistory_Any'].fillna(0).astype(int) +
        input_df['AntidiabeticMedications'].fillna(0).astype(int)
    )
    input_df['Symptom_Score'] = input_df['Edema'].fillna(0) + input_df['UrinaryTractInfections'].fillna(0) + (input_df['NauseaVomiting'].fillna(0) > 0).astype(int)
    input_df['Age_GFR_Interaction'] = input_df['Age'].fillna(50) * (120 - input_df['GFR'].fillna(60)) / 100
    input_df['GFR_ACR_Product'] = input_df['GFR'].fillna(60) * input_df['ACR'].fillna(150) / 1000
    
    input_df = input_df[features]
    pred = loaded_model.predict(input_df)[0]
    probs = loaded_model.predict_proba(input_df)[0]
    levels = ['Low', 'Medium', 'High']
    
    return f"Predicted Risk: {levels[pred]} (Probabilities: Low {probs[0]:.2%}, Medium {probs[1]:.2%}, High {probs[2]:.2%})"

# Test predictions
print("\n🧪 CLINICAL TEST CASES:")
print("="*80)

tests = [
    {'name': 'Young Healthy', 'input': {'Age': 32, 'Gender': 0, 'SystolicBP': 118, 'BMI': 23, 'Smoking': 0, 'AlcoholConsumption': 0, 'PhysicalActivity': 5, 'Edema': 0, 'UrinaryTractInfections': 0, 'NauseaVomiting': 0, 'FamilyHistory_Any': 0, 'AntidiabeticMedications': 0, 'GFR': 105, 'ACR': 15}, 'expected': 'LOW'},
    {'name': 'Middle-aged, Mild HTN', 'input': {'Age': 45, 'Gender': 1, 'SystolicBP': 140, 'BMI': 28.5, 'Smoking': 0, 'AlcoholConsumption': 1, 'PhysicalActivity': 2, 'Edema': 0, 'UrinaryTractInfections': 0, 'NauseaVomiting': 0, 'FamilyHistory_Any': 1, 'AntidiabeticMedications': 0, 'GFR': 72, 'ACR': 20}, 'expected': 'LOW/MED'},
    {'name': 'Stage 3a CKD', 'input': {'Age': 55, 'Gender': 1, 'SystolicBP': 145, 'BMI': 30, 'Smoking': 0, 'AlcoholConsumption': 5, 'PhysicalActivity': 2, 'Edema': 0, 'UrinaryTractInfections': 0, 'NauseaVomiting': 0, 'FamilyHistory_Any': 1, 'AntidiabeticMedications': 1, 'GFR': 52, 'ACR': 85}, 'expected': 'MEDIUM'},
    {'name': 'Stage 4 Advanced', 'input': {'Age': 68, 'Gender': 1, 'SystolicBP': 165, 'BMI': 32, 'Smoking': 1, 'AlcoholConsumption': 15, 'PhysicalActivity': 1, 'Edema': 1, 'UrinaryTractInfections': 1, 'NauseaVomiting': 1, 'FamilyHistory_Any': 1, 'AntidiabeticMedications': 1, 'GFR': 25, 'ACR': 320}, 'expected': 'HIGH'}
]

for i, test in enumerate(tests, 1):
    print(f"\n{i}. {test['name']} (Expected: {test['expected']})")
    print(f"   {predict_risk(test['input'])}")

print("\n" + "="*80)
print("✅ ULTIMATE MODEL COMPLETE!")
print("="*80)

all_pass = min_precision >= 0.85 and min_recall >= 0.85 and min_f1 >= 0.850 and gap <= 0.03
almost_pass = min_precision >= 0.83 and min_recall >= 0.83 and min_f1 >= 0.830 and gap <= 0.05

print(f"\n📊 FINAL SCORECARD:")
print(f"  Test Accuracy:       {test_acc:.1%}")
print(f"  Train-Test Gap:      {gap:.1%}")
print(f"  Min Class Precision: {min_precision:.1%}")
print(f"  Min Class Recall:    {min_recall:.1%}")
print(f"  Min Class F1:        {min_f1:.3f}")
print(f"  CV Stability:        {cv_scores.std():.4f}")
print(f"\n  🎯 STATUS: {'✅✅✅ ALL TARGETS MET!' if all_pass else '✅✅ VERY CLOSE!' if almost_pass else '⚠️ Check details'}")
print(f"  📦 API Compatible: YES (no parameter changes needed)")
print("="*80)

🏥 ULTIMATE CKD RISK MODEL
   (Advanced Feature Engineering + Optimized Hyperparameters)

✓ Dataset loaded: 1659 samples

📊 Risk Distribution:
  Low   :  476 ( 28.7%)
  Medium:  548 ( 33.0%)
  High  :  635 ( 38.3%)

🔧 Engineering Features...
✅ 10 engineered features created
📊 Total features: 29 (19 original + 10 engineered)

📋 Data Split: Train=1244, Test=415

🔄 Training optimized Random Forest...
✅ Training complete!
📊 OOB Score: 0.8963 (89.6%)

📊 MODEL PERFORMANCE:
Training Accuracy:   0.9381 (93.8%)
OOB Accuracy:        0.8963 (89.6%)
Testing Accuracy:    0.9036 (90.4%)
Train-Test Gap:      0.0345 (3.4%)

✅ EXCELLENT: Gap ≤ 5%

🎯 TEST SET RESULTS:
Confusion Matrix:
         Predicted →
Actual ↓   Low  Med  High
Low       109   10     0
Medium      3  117    17
High        0   10   149

--------------------------------------------------------------------------------
              precision    recall  f1-score   support

         Low      0.973     0.916     0.944       119
      Mediu

In [4]:
print(f"\n📊 FINAL SCORECARD:")
print(f"  Test Accuracy:       {test_acc:.1%}")
print(f"  Train-Test Gap:      {gap:.1%}")
print(f"  Min Class Precision: {min_precision:.1%}")
print(f"  Min Class Recall:    {min_recall:.1%}")
print(f"  Min Class F1:        {min_f1:.3f}")
print(f"  CV Stability:        {cv_scores.std():.4f}")
print(f"\n  🎯 STATUS: {'✅✅✅ ALL TARGETS MET!' if all_pass else '✅✅ VERY CLOSE!' if almost_pass else '⚠️ Check details'}")
print(f"  📦 API Compatible: YES (no parameter changes needed)")
print("="*80)


📊 FINAL SCORECARD:
  Test Accuracy:       90.4%
  Train-Test Gap:      3.4%
  Min Class Precision: 85.4%
  Min Class Recall:    85.4%
  Min Class F1:        0.854
  CV Stability:        0.0143

  🎯 STATUS: ✅✅ VERY CLOSE!
  📦 API Compatible: YES (no parameter changes needed)


In [5]:
print("\n🎯 TEST SET RESULTS:")
print("="*80)
cm = confusion_matrix(y_test, y_test_pred)
print("Confusion Matrix:")
print("         Predicted →")
print("Actual ↓   Low  Med  High")
for i, label in enumerate(['Low', 'Medium', 'High']):
    print(f"{label:6s}    {cm[i, 0]:3d}  {cm[i, 1]:3d}   {cm[i, 2]:3d}")

print("\n" + "-"*80)
print(classification_report(
    y_test, y_test_pred, 
    target_names=['Low', 'Medium', 'High'], 
    digits=3
))


🎯 TEST SET RESULTS:
Confusion Matrix:
         Predicted →
Actual ↓   Low  Med  High
Low       109   10     0
Medium      3  117    17
High        0   10   149

--------------------------------------------------------------------------------
              precision    recall  f1-score   support

         Low      0.973     0.916     0.944       119
      Medium      0.854     0.854     0.854       137
        High      0.898     0.937     0.917       159

    accuracy                          0.904       415
   macro avg      0.908     0.902     0.905       415
weighted avg      0.905     0.904     0.904       415

